# Carga de datos procesados

Por ahora este notebook solo carga las tablas procesadas de ENDES. No filtra, no une, no crea el target y no modifica datos.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
# Ubica la raíz del proyecto al ejecutar desde modelo/ o desde la raíz.
rutas_posibles = [Path.cwd(), Path.cwd().parent]
RAIZ = next((ruta for ruta in rutas_posibles if (ruta / 'datos_procesados').exists()), None)

if RAIZ is None:
    raise FileNotFoundError('No se encontró la carpeta datos_procesados/.')

RUTA_DATOS = RAIZ / 'datos_procesados'
RUTA_DATOS

In [ ]:
tabla_2024 = pd.read_csv(RUTA_DATOS / 'tabla_nino_2024.csv')
tabla_2025 = pd.read_csv(RUTA_DATOS / 'tabla_nino_2025.csv')

print(f'2024: {tabla_2024.shape[0]:,} filas y {tabla_2024.shape[1]} columnas')
print(f'2025: {tabla_2025.shape[0]:,} filas y {tabla_2025.shape[1]} columnas')

In [ ]:
tabla_2025.head()

# 1629 2025 procesamiento

In [ ]:
from pathlib import Path
import pandas as pd

CARPETA_1629_2025 = (
    RAIZ
    / "modelo"
    / "datos"
    / "2025"
    / "1038-Modulo1629"
    / "1036-Modulo1629"
)

rutas_csv_1629 = sorted(CARPETA_1629_2025.glob("*.csv"))

if not rutas_csv_1629:
    raise FileNotFoundError(
        f"No se encontraron CSV en: {CARPETA_1629_2025}"
    )

# La clave conserva el código/nombre del CSV.
tablas_1629 = {
    ruta.stem: pd.read_csv(
        ruta,
        encoding="utf-8-sig",
        dtype=str
    )
    for ruta in rutas_csv_1629
}

for codigo_csv, tabla in tablas_1629.items():
    print(f"{codigo_csv}: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")

In [ ]:
# Recargamos HHID del niño como texto para preservar ceros iniciales.
ids_2025 = pd.read_csv(
    RUTA_DATOS / "tabla_nino_2025.csv",
    usecols=["caseid", "hhid"],
    dtype="string"
)

hhid_ninos = ids_2025["hhid"].str.strip()

resultados_hhid = []

for codigo_csv, tabla in tablas_1629.items():

    if "HHID" not in tabla.columns:
        print(f"{codigo_csv}: no tiene la columna HHID")
        continue

    # HHID únicos presentes en cada CSV del módulo 1629.
    hhid_tabla = (
        tabla["HHID"]
        .astype("string")
        .str.strip()
        .dropna()
        .unique()
    )

    ninos_con_coincidencia = hhid_ninos.isin(set(hhid_tabla))

    resultados_hhid.append({
        "csv": codigo_csv,
        "filas_csv": len(tabla),
        "hogares_unicos_csv": len(hhid_tabla),
        "ninos_2025": len(ids_2025),
        "ninos_con_hhid": ninos_con_coincidencia.sum(),
        "ninos_sin_hhid": (~ninos_con_coincidencia).sum(),
        "porcentaje_coincidencia": ninos_con_coincidencia.mean()
    })

resumen_hhid = pd.DataFrame(resultados_hhid)

resumen_hhid.style.format({
    "porcentaje_coincidencia": "{:.2%}"
})

In [ ]:
for codigo_csv, tabla in tablas_1629.items():
    print(f"\n{'=' * 70}")
    print(f"{codigo_csv} — {tabla.shape[1]} columnas")
    print("=" * 70)

    for numero, columna in enumerate(tabla.columns, start=1):
        print(f"{numero:>2}. {columna}")

In [ ]:
# -------------------------------------------------------------------
# Tabla de características del hogar — módulo 1629, año 2025
# Una fila por HHID. Todavía NO se une a tabla_2025.
# -------------------------------------------------------------------

# RECH0: información resumida del hogar.
rech0 = tablas_1629["RECH0_2025"][
    ["HHID", "HV012", "HV014"]
].copy()

rech0 = rech0.rename(columns={
    "HHID": "hhid",
    "HV012": "n_residentes_habituales",
    "HV014": "n_menores_5_hogar",
})

rech0["hhid"] = rech0["hhid"].astype("string").str.strip()

for columna in ["n_residentes_habituales", "n_menores_5_hogar"]:
    rech0[columna] = pd.to_numeric(rech0[columna], errors="coerce")

# Debe haber una sola fila de RECH0 por hogar.
if rech0["hhid"].duplicated().any():
    raise ValueError("RECH0 tiene HHID duplicados; revisar antes de unir.")

# RECH1: integrantes del hogar.
rech1 = tablas_1629["RECH1_2025"][
    ["HHID", "HV102", "HV105"]
].copy()

rech1["hhid"] = rech1["HHID"].astype("string").str.strip()
rech1["residente_habitual"] = pd.to_numeric(
    rech1["HV102"],
    errors="coerce"
)
rech1["edad_integrante"] = pd.to_numeric(
    rech1["HV105"],
    errors="coerce"
)

# Solo integrantes que viven habitualmente en el hogar.
rech1_residentes = rech1.loc[
    rech1["residente_habitual"].eq(1)
].copy()

# Agregados de composición familiar, una fila por hogar.
composicion_hogar = (
    rech1_residentes
    .groupby("hhid", as_index=False)
    .agg(
        n_ninos_6_17_hogar=(
            "edad_integrante",
            lambda edad: edad.between(6, 17).sum()
        ),
        n_adultos_18_59_hogar=(
            "edad_integrante",
            lambda edad: edad.between(18, 59).sum()
        ),
        n_adultos_mayores_hogar=(
            "edad_integrante",
            lambda edad: edad.ge(60).sum()
        ),
    )
)

# Tabla final del módulo 1629 para unir luego con tabla_2025.
hogar_1629_2025 = rech0.merge(
    composicion_hogar,
    on="hhid",
    how="left",
    validate="one_to_one"
)

# Si un hogar no tiene integrantes válidos en RECH1, queda en cero.
columnas_conteo = [
    "n_ninos_6_17_hogar",
    "n_adultos_18_59_hogar",
    "n_adultos_mayores_hogar",
]

hogar_1629_2025[columnas_conteo] = (
    hogar_1629_2025[columnas_conteo]
    .fillna(0)
    .astype("Int64")
)

print(f"Hogares en la tabla final: {len(hogar_1629_2025):,}")
print(f"HHID únicos: {hogar_1629_2025['hhid'].nunique():,}")

hogar_1629_2025.head()

# 1630 2025 procesamiento

In [ ]:
CARPETA_1630_2025 = (
    RAIZ
    / "modelo"
    / "datos"
    / "2025"
    / "1038-Modulo1630"
    / "1036-Modulo1630"
)

rutas_csv_1630 = sorted(CARPETA_1630_2025.glob("*.csv"))

if not rutas_csv_1630:
    raise FileNotFoundError(
        f"No se encontraron CSV en: {CARPETA_1630_2025}"
    )

# Clave = nombre/código del CSV, por ejemplo: RECH23_2025
tablas_1630 = {
    ruta.stem: pd.read_csv(
        ruta,
        encoding="utf-8-sig",
        dtype=str
    )
    for ruta in rutas_csv_1630
}

for codigo_csv, tabla in tablas_1630.items():
    print(f"{codigo_csv}: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
tablas_1630["RECH23_2025"].head()

In [ ]:
# Recargamos HHID del niño como texto para preservar ceros iniciales.
ids_2025 = pd.read_csv(
    RUTA_DATOS / "tabla_nino_2025.csv",
    usecols=["caseid", "hhid"],
    dtype="string"
)

hhid_ninos = ids_2025["hhid"].str.strip()

resultados_hhid = []

for codigo_csv, tabla in tablas_1630.items():

    if "HHID" not in tabla.columns:
        print(f"{codigo_csv}: no tiene la columna HHID")
        continue

    # HHID únicos presentes en cada CSV del módulo 1629.
    hhid_tabla = (
        tabla["HHID"]
        .astype("string")
        .str.strip()
        .dropna()
        .unique()
    )

    ninos_con_coincidencia = hhid_ninos.isin(set(hhid_tabla))

    resultados_hhid.append({
        "csv": codigo_csv,
        "filas_csv": len(tabla),
        "hogares_unicos_csv": len(hhid_tabla),
        "ninos_2025": len(ids_2025),
        "ninos_con_hhid": ninos_con_coincidencia.sum(),
        "ninos_sin_hhid": (~ninos_con_coincidencia).sum(),
        "porcentaje_coincidencia": ninos_con_coincidencia.mean()
    })

resumen_hhid = pd.DataFrame(resultados_hhid)

resumen_hhid.style.format({
    "porcentaje_coincidencia": "{:.2%}"
})

In [ ]:
for codigo_csv, tabla in tablas_1630.items():
    print(f"\n{'=' * 70}")
    print(f"{codigo_csv} — {tabla.shape[1]} columnas")
    print("=" * 70)

    for numero, columna in enumerate(tabla.columns, start=1):
        print(f"{numero:>2}. {columna}")

In [ ]:
# -------------------------------------------------------------------
# Tabla de vivienda — módulo 1630, año 2025
# Una fila por HHID.
# Por ahora solo: riqueza y hacinamiento.
# -------------------------------------------------------------------

rech23 = tablas_1630["RECH23_2025"][
    ["HHID", "HV270", "HV216"]
].copy()

rech23 = rech23.rename(columns={
    "HHID": "hhid",
    "HV270": "quintil_riqueza",
    "HV216": "n_habitaciones_dormir",
})

rech23["hhid"] = rech23["hhid"].astype("string").str.strip()

rech23["quintil_riqueza"] = pd.to_numeric(
    rech23["quintil_riqueza"],
    errors="coerce"
).astype("Int64")

rech23["n_habitaciones_dormir"] = pd.to_numeric(
    rech23["n_habitaciones_dormir"],
    errors="coerce"
)

# Cero habitaciones o valores negativos no sirven para calcular hacinamiento.
rech23.loc[
    rech23["n_habitaciones_dormir"].le(0),
    "n_habitaciones_dormir"
] = pd.NA

# Debe haber una fila de RECH23 por hogar.
if rech23["hhid"].duplicated().any():
    raise ValueError("RECH23 tiene HHID duplicados; revisar antes de unir.")

# Se agrega el número de residentes habituales que ya creamos desde RECH0.
vivienda_1630_2025 = rech23.merge(
    hogar_1629_2025[["hhid", "n_residentes_habituales"]],
    on="hhid",
    how="left",
    validate="one_to_one"
)

# Hacinamiento: residentes habituales por habitación usada para dormir.
vivienda_1630_2025["personas_por_dormitorio"] = (
    vivienda_1630_2025["n_residentes_habituales"]
    / vivienda_1630_2025["n_habitaciones_dormir"]
)

# Dejamos las columnas finales, listas para unir luego con tabla_2025.
vivienda_1630_2025 = vivienda_1630_2025[
    [
        "hhid",
        "quintil_riqueza",
        "n_habitaciones_dormir",
        "personas_por_dormitorio",
    ]
]

print(f"Hogares en vivienda_1630_2025: {len(vivienda_1630_2025):,}")
print(f"HHID únicos: {vivienda_1630_2025['hhid'].nunique():,}")

vivienda_1630_2025.head()

# 1631 2025 procesamiento

In [ ]:
CARPETA_1631_2025 = (
    RAIZ
    / "modelo"
    / "datos"
    / "2025"
    / "1038-Modulo1631"
    / "1036-Modulo1631"
)

rutas_csv_1631 = sorted(CARPETA_1631_2025.glob("*.csv"))

if not rutas_csv_1631:
    raise FileNotFoundError(
        f"No se encontraron CSV en: {CARPETA_1631_2025}"
    )

# Clave = nombre/código del CSV, por ejemplo: RECH23_2025
tablas_1631 = {
    ruta.stem: pd.read_csv(
        ruta,
        encoding="utf-8-sig",
        dtype=str
    )
    for ruta in rutas_csv_1631
}

for codigo_csv, tabla in tablas_1631.items():
    print(f"{codigo_csv}: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
    

In [ ]:
# Recargamos CASEID como texto para no perder ceros iniciales.
ids_2025 = pd.read_csv(
    RUTA_DATOS / "tabla_nino_2025.csv",
    usecols=["caseid"],
    dtype="string"
)

caseid_ninos = ids_2025["caseid"].str.strip()

resultados_caseid = []

for codigo_csv, tabla in tablas_1631.items():

    if "CASEID" not in tabla.columns:
        print(f"{codigo_csv}: no tiene la columna CASEID")
        continue

    caseid_tabla = (
        tabla["CASEID"]
        .astype("string")
        .str.strip()
    )

    caseid_validos = caseid_tabla.dropna()
    caseid_unicos = caseid_validos.drop_duplicates()

    ninos_con_coincidencia = caseid_ninos.isin(set(caseid_unicos))

    resultados_caseid.append({
        "csv": codigo_csv,
        "filas_csv": len(tabla),
        "caseid_unicos_csv": len(caseid_unicos),
        "caseid_duplicados_csv": caseid_validos.duplicated().sum(),
        "ninos_2025": len(ids_2025),
        "ninos_con_caseid": ninos_con_coincidencia.sum(),
        "ninos_sin_caseid": (~ninos_con_coincidencia).sum(),
        "porcentaje_coincidencia": ninos_con_coincidencia.mean()
    })

resumen_caseid = pd.DataFrame(resultados_caseid)

display(
    resumen_caseid.style.format({
        "porcentaje_coincidencia": "{:.2%}"
    })
)

In [ ]:
for codigo_csv, tabla in tablas_1631.items():
    print(f"\n{'=' * 70}")
    print(f"{codigo_csv} — {tabla.shape[1]} columnas")
    print("=" * 70)

    for numero, columna in enumerate(tabla.columns, start=1):
        print(f"{numero:>2}. {columna}")

In [ ]:
# -------------------------------------------------------------------
# Tabla materna — módulo 1631, año 2025
# Una fila por CASEID.
# Todavía NO se une con tabla_2025.
# -------------------------------------------------------------------

# Datos básicos de la madre.
madre_rec0111 = tablas_1631["REC0111_2025"][
    ["CASEID", "V012", "V149"]
].copy()

madre_rec0111 = madre_rec0111.rename(columns={
    "CASEID": "caseid",
    "V012": "edad_madre",
    "V149": "nivel_educativo_madre",
})

# Datos sociales y de cuidado.
madre_rec91 = tablas_1631["REC91_2025"][
    ["CASEID", "S229B1", "S720A", "S119"]
].copy()

madre_rec91 = madre_rec91.rename(columns={
    "CASEID": "caseid",
    "S229B1": "madre_afiliada_sis",
    "S720A": "cuidador_habitual",
    "S119": "lengua_materna",
})

# CASEID siempre como texto.
for tabla in [madre_rec0111, madre_rec91]:
    tabla["caseid"] = tabla["caseid"].astype("string").str.strip()

# Validamos que haya una fila por madre en cada CSV.
if madre_rec0111["caseid"].duplicated().any():
    raise ValueError("REC0111 tiene CASEID duplicados.")

if madre_rec91["caseid"].duplicated().any():
    raise ValueError("REC91 tiene CASEID duplicados.")

# Unimos las dos fuentes de datos maternos.
materna_1631_2025 = madre_rec0111.merge(
    madre_rec91,
    on="caseid",
    how="left",
    validate="one_to_one"
)

# Nos quedamos solo con las madres de niños presentes en tabla_2025.
caseid_ninos_2025 = set(ids_2025["caseid"].dropna().str.strip())

materna_1631_2025 = materna_1631_2025.loc[
    materna_1631_2025["caseid"].isin(caseid_ninos_2025)
].copy()

# Edad como número; las demás columnas se conservan como categorías/códigos.
materna_1631_2025["edad_madre"] = pd.to_numeric(
    materna_1631_2025["edad_madre"],
    errors="coerce"
)

print(f"Madres en la tabla final: {len(materna_1631_2025):,}")
print(f"CASEID únicos: {materna_1631_2025['caseid'].nunique():,}")

materna_1631_2025.head()

# 1634 2025 procesamiento

In [ ]:
CARPETA_1634_2025 = (
    RAIZ
    / "modelo"
    / "datos"
    / "2025"
    / "1038-Modulo1634"
    / "1036-Modulo1634"
)

rutas_csv_1634 = sorted(CARPETA_1634_2025.glob("*.csv"))

if not rutas_csv_1634:
    raise FileNotFoundError(
        f"No se encontraron CSV en: {CARPETA_1634_2025}"
    )

# Clave = nombre/código del CSV, por ejemplo: RECH23_2025
tablas_1634 = {
    ruta.stem: pd.read_csv(
        ruta,
        encoding="utf-8-sig",
        dtype=str
    )
    for ruta in rutas_csv_1634
}

for codigo_csv, tabla in tablas_1634.items():
    print(f"{codigo_csv}: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
    

In [ ]:
for codigo_csv, tabla in tablas_1634.items():
    print(f"\n{'=' * 70}")
    print(f"{codigo_csv} — {tabla.shape[1]} columnas")
    print("=" * 70)

    for numero, columna in enumerate(tabla.columns, start=1):
        print(f"{numero:>2}. {columna}")

In [ ]:
# Recargamos HHID del niño como texto para preservar ceros iniciales.
ids_2025 = pd.read_csv(
    RUTA_DATOS / "tabla_nino_2025.csv",
    usecols=["caseid", "hhid"],
    dtype="string"
)

hhid_ninos = ids_2025["hhid"].str.strip()

resultados_hhid = []

for codigo_csv, tabla in tablas_1634.items():

    if "HHID" not in tabla.columns:
        print(f"{codigo_csv}: no tiene la columna HHID")
        continue

    # HHID únicos presentes en cada CSV del módulo 1629.
    hhid_tabla = (
        tabla["HHID"]
        .astype("string")
        .str.strip()
        .dropna()
        .unique()
    )

    ninos_con_coincidencia = hhid_ninos.isin(set(hhid_tabla))

    resultados_hhid.append({
        "csv": codigo_csv,
        "filas_csv": len(tabla),
        "hogares_unicos_csv": len(hhid_tabla),
        "ninos_2025": len(ids_2025),
        "ninos_con_hhid": ninos_con_coincidencia.sum(),
        "ninos_sin_hhid": (~ninos_con_coincidencia).sum(),
        "porcentaje_coincidencia": ninos_con_coincidencia.mean()
    })

resumen_hhid = pd.DataFrame(resultados_hhid)

resumen_hhid.style.format({
    "porcentaje_coincidencia": "{:.2%}"
})

In [ ]:
# 3. Normalizar las llaves antes de comparar.
# CASEID identifica a la madre/entrevista; IDX95 identifica al niño dentro de ella.

def normalizar_caseid(serie):
    return serie.astype("string").str.strip()

def normalizar_indice(serie):
    return (
        pd.to_numeric(
            serie.astype("string")
                 .str.strip()
                 .str.replace(r"\.0$", "", regex=True),
            errors="coerce"
        )
        .astype("Int64")
    )

llaves_ninos_2025 = tabla_2025[["caseid", "idx_nino"]].copy()

llaves_ninos_2025["_caseid"] = normalizar_caseid(llaves_ninos_2025["caseid"])
llaves_ninos_2025["_idx95"] = normalizar_indice(llaves_ninos_2025["idx_nino"])

print("Niños en tabla_2025:", len(llaves_ninos_2025))
print(
    "Llaves CASEID + IDX95 duplicadas en tabla_2025:",
    llaves_ninos_2025.duplicated(["_caseid", "_idx95"]).sum()
)

In [ ]:
# 4. Validar específicamente el cruce con REC95.
# Este es el archivo prioritario del módulo 1634.

rec95 = tablas_1634["REC95_2025"].copy()

columnas_llave = {"CASEID", "IDX95"}
faltantes = columnas_llave - set(rec95.columns)

if faltantes:
    raise KeyError(f"REC95 no tiene estas columnas necesarias: {faltantes}")

llaves_rec95 = rec95[["CASEID", "IDX95"]].copy()

llaves_rec95["_caseid"] = normalizar_caseid(llaves_rec95["CASEID"])
llaves_rec95["_idx95"] = normalizar_indice(llaves_rec95["IDX95"])

llaves_rec95_validas = (
    llaves_rec95
    .dropna(subset=["_caseid", "_idx95"])
    .drop_duplicates(["_caseid", "_idx95"])
)

verificacion_rec95 = llaves_ninos_2025.merge(
    llaves_rec95_validas[["_caseid", "_idx95"]].assign(en_rec95=1),
    on=["_caseid", "_idx95"],
    how="left"
)

ninos_en_rec95 = verificacion_rec95["en_rec95"].eq(1)

resumen_cruce_rec95 = pd.DataFrame([{
    "ninos_2025": len(llaves_ninos_2025),
    "registros_REC95": len(rec95),
    "llaves_unicas_REC95": len(llaves_rec95_validas),
    "llaves_duplicadas_REC95": (
        llaves_rec95.duplicated(["_caseid", "_idx95"]).sum()
    ),
    "ninos_con_REC95": ninos_en_rec95.sum(),
    "ninos_sin_REC95": (~ninos_en_rec95).sum(),
    "porcentaje_coincidencia": ninos_en_rec95.mean()
}])

resumen_cruce_rec95.style.format({
    "porcentaje_coincidencia": "{:.2%}"
})

In [ ]:
# 5. Ver las variables potenciales de CRED y entregas de hierro.
# Aún NO se usarán como predictores: primero debemos revisar fechas
# para evitar usar información posterior a la medición del target.

variables_candidatas_cred = [
    columna
    for columna in rec95.columns
    if columna.startswith("S466") or columna.startswith("Q465EC")
]

print("Variables candidatas relacionadas con CRED o entregas:")
for columna in variables_candidatas_cred:
    print("-", columna)

In [ ]:
# 6. Crear una tabla de revisión, sin target ni variables de toma semanal.
# Esta tabla todavía NO se une al modelo.

cred_1634_revision = rec95[
    ["CASEID", "IDX95"] + variables_candidatas_cred
].copy()

cred_1634_revision = cred_1634_revision.rename(columns={
    "CASEID": "caseid",
    "IDX95": "idx_nino"
})

cred_1634_revision.head()

In [ ]:
# Tabla de variables candidatas desde REC95:
# CRED + entregas de micronutrientes.
# No incluye S465EA-ED, S465DB_*, S465DD_*, Q465DF, Q465DH, Q465DJ ni Q465EB_A8.

rec95 = tablas_1634["REC95_2025"].copy()

columnas_necesarias = [
    "CASEID", "IDX95",
    "S466", "S466AA", "S466AB", "S466AC",
    "S466AD", "S466AE", "S466AX", "S466B",
    "Q465EC_A9"
]

faltantes = set(columnas_necesarias) - set(rec95.columns)

if faltantes:
    raise KeyError(f"Faltan columnas en REC95: {sorted(faltantes)}")


def a_numero(serie):
    return pd.to_numeric(
        serie.astype("string").str.strip(),
        errors="coerce"
    )


def crear_fecha(tabla, columna_dia, columna_mes, columna_anio):
    """Crea fechas válidas; códigos como 44, 98 o 9998 quedan como NaT."""

    dia = a_numero(tabla[columna_dia])
    mes = a_numero(tabla[columna_mes])
    anio = a_numero(tabla[columna_anio])

    fecha = pd.to_datetime(
        pd.DataFrame({
            "year": anio,
            "month": mes,
            "day": dia
        }),
        errors="coerce"
    )

    es_fecha_valida = (
        dia.between(1, 31)
        & mes.between(1, 12)
        & anio.between(2000, 2025)
    )

    return fecha.where(es_fecha_valida)


# Fechas de los 12 controles CRED registrados.
fechas_cred = pd.DataFrame(index=rec95.index)

for numero_control in range(1, 13):
    fechas_cred[f"cred_{numero_control}"] = crear_fecha(
        rec95,
        f"S466DC{numero_control}",
        f"S466MC{numero_control}",
        f"S466YC{numero_control}"
    )


# En el diccionario existe un aparente error de nombre:
# Q645EC_Y2, en vez de Q465EC_Y2.
columna_anio_entrega_2 = (
    "Q465EC_Y2"
    if "Q465EC_Y2" in rec95.columns
    else "Q645EC_Y2"
)

anios_entregas = {
    1: "Q465EC_Y1",
    2: columna_anio_entrega_2,
    3: "Q465EC_Y3",
    4: "Q465EC_Y4",
    5: "Q465EC_Y5",
    6: "Q465EC_Y6"
}

# Fechas de hasta seis entregas de micronutrientes registradas.
fechas_entregas_mmn = pd.DataFrame(index=rec95.index)

for numero_entrega in range(1, 7):
    fechas_entregas_mmn[f"entrega_mmn_{numero_entrega}"] = crear_fecha(
        rec95,
        f"Q465EC_D{numero_entrega}",
        f"Q465EC_M{numero_entrega}",
        anios_entregas[numero_entrega]
    )


# Q465EC_A9:
# 1-6 = número de entregas
# 7 = carné sin fecha
# 8 = no muestra carné
codigo_entregas_mmn = a_numero(rec95["Q465EC_A9"])

cred_1634_2025 = pd.DataFrame({
    "caseid": normalizar_caseid(rec95["CASEID"]),
    "idx_nino": normalizar_indice(rec95["IDX95"]),

    # CRED
    "tuvo_cred": a_numero(rec95["S466"]).map({
        0: "no",
        1: "si",
        8: "no_sabe"
    }),
    "cred_realizado_por_doctor": a_numero(rec95["S466AA"]),
    "cred_realizado_por_obstetra": a_numero(rec95["S466AB"]),
    "cred_realizado_por_enfermera": a_numero(rec95["S466AC"]),
    "cred_realizado_por_tecnico_enfermeria": a_numero(rec95["S466AD"]),
    "cred_realizado_por_promotor_salud": a_numero(rec95["S466AE"]),
    "cred_realizado_por_otro": a_numero(rec95["S466AX"]),
    "lugar_cred_codigo": a_numero(rec95["S466B"]),

    # Historial CRED con fecha válida
    "n_cred_con_fecha": fechas_cred.notna().sum(axis=1).astype("Int64"),
    "fecha_ultimo_cred": fechas_cred.max(axis=1),

    # Entregas de micronutrientes
    "n_entregas_mmn_registradas": codigo_entregas_mmn.where(
        codigo_entregas_mmn.between(1, 6)
    ).astype("Int64"),
    "registro_entrega_mmn": codigo_entregas_mmn.map({
        1: "1_entrega",
        2: "2_entregas",
        3: "3_entregas",
        4: "4_entregas",
        5: "5_entregas",
        6: "6_entregas",
        7: "carne_sin_fecha",
        8: "no_muestra_carne"
    }),
    "n_entregas_mmn_con_fecha": (
        fechas_entregas_mmn.notna().sum(axis=1).astype("Int64")
    ),
    "fecha_ultima_entrega_mmn": fechas_entregas_mmn.max(axis=1)
})

# Una fila por niño: condición necesaria antes de unirla a tabla_2025.
if cred_1634_2025.duplicated(["caseid", "idx_nino"]).any():
    raise ValueError("Hay llaves CASEID + IDX95 duplicadas en la tabla generada.")

cred_1634_2025.head()

# 1641 2025 procesamiento

In [ ]:
CARPETA_1641_2025 = (
    RAIZ
    / "modelo"
    / "datos"
    / "2025"
    / "1038-Modulo1641"
    / "1036-Modulo1641"
)

rutas_csv_1641 = sorted(CARPETA_1641_2025.glob("*.csv"))

if not rutas_csv_1641:
    raise FileNotFoundError(
        f"No se encontraron CSV en: {CARPETA_1641_2025}"
    )

# Clave = nombre/código del CSV, por ejemplo: RECH23_2025
tablas_1641 = {
    ruta.stem: pd.read_csv(
        ruta,
        encoding="utf-8-sig",
        dtype=str
    )
    for ruta in rutas_csv_1641
}

for codigo_csv, tabla in tablas_1641.items():
    print(f"{codigo_csv}: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
    

In [ ]:
for codigo_csv, tabla in tablas_1641.items():
    print(f"\n{'=' * 70}")
    print(f"{codigo_csv} — {tabla.shape[1]} columnas")
    print("=" * 70)

    for numero, columna in enumerate(tabla.columns, start=1):
        print(f"{numero:>2}. {columna}")

In [ ]:
# Recargamos HHID del niño como texto para preservar ceros iniciales.
ids_2025 = pd.read_csv(
    RUTA_DATOS / "tabla_nino_2025.csv",
    usecols=["caseid", "hhid"],
    dtype="string"
)

hhid_ninos = ids_2025["hhid"].str.strip()

resultados_hhid = []

for codigo_csv, tabla in tablas_1641.items():

    if "HHID" not in tabla.columns:
        print(f"{codigo_csv}: no tiene la columna HHID")
        continue

    # HHID únicos presentes en cada CSV del módulo 1629.
    hhid_tabla = (
        tabla["HHID"]
        .astype("string")
        .str.strip()
        .dropna()
        .unique()
    )

    ninos_con_coincidencia = hhid_ninos.isin(set(hhid_tabla))

    resultados_hhid.append({
        "csv": codigo_csv,
        "filas_csv": len(tabla),
        "hogares_unicos_csv": len(hhid_tabla),
        "ninos_2025": len(ids_2025),
        "ninos_con_hhid": ninos_con_coincidencia.sum(),
        "ninos_sin_hhid": (~ninos_con_coincidencia).sum(),
        "porcentaje_coincidencia": ninos_con_coincidencia.mean()
    })

resumen_hhid = pd.DataFrame(resultados_hhid)

resumen_hhid.style.format({
    "porcentaje_coincidencia": "{:.2%}"
})

# generacion de nuevo tabla2025

In [ ]:
# ============================================================
# UNIÓN FINAL 2025: tabla principal + módulos 1629, 1630, 1631 y 1634
# ============================================================

def normalizar_texto_id(serie):
    return serie.astype("string").str.strip()


def normalizar_indice_id(serie):
    return (
        pd.to_numeric(
            serie.astype("string")
                 .str.strip()
                 .str.replace(r"\.0$", "", regex=True),
            errors="coerce"
        )
        .astype("Int64")
    )


def verificar_llave_unica(tabla, llaves, nombre_tabla):
    duplicados = tabla.duplicated(llaves).sum()

    if duplicados > 0:
        raise ValueError(
            f"{nombre_tabla} tiene {duplicados:,} llaves duplicadas "
            f"en {llaves}."
        )


def unir_tabla(base, adicional, llaves, nombre_tabla):
    columnas_repetidas = (
        set(base.columns)
        .intersection(adicional.columns)
        .difference(llaves)
    )

    if columnas_repetidas:
        raise ValueError(
            f"{nombre_tabla} tiene columnas ya existentes en la tabla principal: "
            f"{sorted(columnas_repetidas)}"
        )

    return base.merge(
        adicional,
        on=llaves,
        how="left",
        validate="many_to_one"
    )


# ------------------------------------------------------------
# 1. Copias para no alterar las tablas originales
# ------------------------------------------------------------

tabla_modelo_2025 = tabla_2025.copy()

hogar_1629_union = hogar_1629_2025.copy()
vivienda_1630_union = vivienda_1630_2025.copy()
materna_1631_union = materna_1631_2025.copy()
cred_1634_union = cred_1634_2025.copy()


# ------------------------------------------------------------
# 2. Normalizar identificadores
# ------------------------------------------------------------

# Tabla principal
tabla_modelo_2025["hhid"] = normalizar_texto_id(
    tabla_modelo_2025["hhid"]
)
tabla_modelo_2025["caseid"] = normalizar_texto_id(
    tabla_modelo_2025["caseid"]
)
tabla_modelo_2025["idx_nino"] = normalizar_indice_id(
    tabla_modelo_2025["idx_nino"]
)

# Módulo 1629: hogar
hogar_1629_union["hhid"] = normalizar_texto_id(
    hogar_1629_union["hhid"]
)

# Módulo 1630: vivienda
vivienda_1630_union["hhid"] = normalizar_texto_id(
    vivienda_1630_union["hhid"]
)

# Módulo 1631: madre/cuidadora
materna_1631_union["caseid"] = normalizar_texto_id(
    materna_1631_union["caseid"]
)

# Módulo 1634: niño
cred_1634_union["caseid"] = normalizar_texto_id(
    cred_1634_union["caseid"]
)
cred_1634_union["idx_nino"] = normalizar_indice_id(
    cred_1634_union["idx_nino"]
)


# ------------------------------------------------------------
# 3. Verificar que cada tabla adicional tenga una fila por llave
# ------------------------------------------------------------

verificar_llave_unica(
    hogar_1629_union,
    ["hhid"],
    "hogar_1629_2025"
)

verificar_llave_unica(
    vivienda_1630_union,
    ["hhid"],
    "vivienda_1630_2025"
)

verificar_llave_unica(
    materna_1631_union,
    ["caseid"],
    "materna_1631_2025"
)

verificar_llave_unica(
    cred_1634_union,
    ["caseid", "idx_nino"],
    "cred_1634_2025"
)


# ------------------------------------------------------------
# 4. Unir módulos
# ------------------------------------------------------------

tabla_modelo_2025 = unir_tabla(
    tabla_modelo_2025,
    hogar_1629_union,
    ["hhid"],
    "hogar_1629_2025"
)

tabla_modelo_2025 = unir_tabla(
    tabla_modelo_2025,
    vivienda_1630_union,
    ["hhid"],
    "vivienda_1630_2025"
)

tabla_modelo_2025 = unir_tabla(
    tabla_modelo_2025,
    materna_1631_union,
    ["caseid"],
    "materna_1631_2025"
)

tabla_modelo_2025 = unir_tabla(
    tabla_modelo_2025,
    cred_1634_union,
    ["caseid", "idx_nino"],
    "cred_1634_2025"
)


# ------------------------------------------------------------
# 5. Resultado
# ------------------------------------------------------------

print(f"Filas originales tabla_2025: {len(tabla_2025):,}")
print(f"Filas tabla_modelo_2025:    {len(tabla_modelo_2025):,}")
print(f"Columnas originales:         {tabla_2025.shape[1]:,}")
print(f"Columnas finales:            {tabla_modelo_2025.shape[1]:,}")

tabla_modelo_2025.head()

In [ ]:
# Guardar la tabla integrada de 2025 en datos_procesados

RUTA_SALIDA_2025 = (
    RAIZ
    / "datos_procesados"
    / "tabla_modelo_2025.csv"
)

tabla_modelo_2025.to_csv(
    RUTA_SALIDA_2025,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo guardado en:\n{RUTA_SALIDA_2025}")
print(f"Filas: {len(tabla_modelo_2025):,}")
print(f"Columnas: {tabla_modelo_2025.shape[1]:,}")